In [1]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Keras ile ANN

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

X,y = load_breast_cancer(return_X_y=True)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [3]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16,activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(8,activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1,activation="sigmoid"),
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 641 (2.50 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

In [5]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [6]:
history = model.fit(X_train,y_train,validation_split=0.2,epochs=100,batch_size=16,verbose=2,callbacks=[early_stopping])

Epoch 1/100
23/23 - 4s - 194ms/step - accuracy: 0.4835 - loss: 0.7555 - val_accuracy: 0.8352 - val_loss: 0.5425
Epoch 2/100
23/23 - 1s - 25ms/step - accuracy: 0.7143 - loss: 0.5659 - val_accuracy: 0.9451 - val_loss: 0.4183
Epoch 3/100
23/23 - 0s - 19ms/step - accuracy: 0.8352 - loss: 0.4481 - val_accuracy: 0.9451 - val_loss: 0.3412
Epoch 4/100
23/23 - 1s - 26ms/step - accuracy: 0.8599 - loss: 0.3733 - val_accuracy: 0.9560 - val_loss: 0.2792
Epoch 5/100
23/23 - 0s - 18ms/step - accuracy: 0.8984 - loss: 0.3143 - val_accuracy: 0.9560 - val_loss: 0.2323
Epoch 6/100
23/23 - 0s - 15ms/step - accuracy: 0.8819 - loss: 0.2944 - val_accuracy: 0.9560 - val_loss: 0.1996
Epoch 7/100
23/23 - 0s - 20ms/step - accuracy: 0.9148 - loss: 0.2340 - val_accuracy: 0.9670 - val_loss: 0.1753
Epoch 8/100
23/23 - 1s - 30ms/step - accuracy: 0.9313 - loss: 0.2369 - val_accuracy: 0.9670 - val_loss: 0.1583
Epoch 9/100
23/23 - 1s - 46ms/step - accuracy: 0.9148 - loss: 0.2158 - val_accuracy: 0.9670 - val_loss: 0.1446


In [7]:
test_loss,test_acc = model.evaluate(X_test,y_test,verbose=0)
print("Test kaybı:", test_loss)
print("Test doğruluğu:", test_acc)

Test kaybı: 0.06894318014383316
Test doğruluğu: 0.9736841917037964


#### aynısı PyTorch ile

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader,random_split

In [9]:
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

In [11]:
full_ds = TensorDataset(X_train_tensor,y_train_tensor)
n_train = int(0.8*len(full_ds))
train_ds,val_ds = random_split(full_ds,[n_train,len(full_ds)-n_train])

train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=32)

In [12]:
class ANN(nn.Module):
    def __init__(self,n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features,16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16,1),
        )

    def forward(self,x):
        return self.net(x)

model = ANN(X_train.shape[1]).to(device)
print(model)

ANN(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [13]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [14]:
def accuracy(logits,targets):
    preds = (torch.sigmoid(logits)>0.5).float()
    return (preds==targets).float().mean()

In [15]:
EPOCHS = 20

for epoch in range(1,EPOCHS+1):
    model.train()
    train_loss,train_correct = 0.0,0.0
    for data,targets in train_loader:
        data,targets = data.to(device),targets.to(device)
        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits,targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()*data.size(0)
        train_correct += accuracy(logits,targets).item()*data.size(0)
    train_loss /= len(train_loader.dataset)
    train_acc = train_correct/len(train_loader.dataset)

    model.eval()
    val_loss,val_correct = 0.0,0.0
    with torch.no_grad():
        for data,targets in val_loader:
            data,targets = data.to(device),targets.to(device)
            logits = model(data)
            val_loss += criterion(logits,targets).item()*data.size(0)
            val_correct += accuracy(logits,targets).item()*data.size(0)
    val_loss /= len(val_loader.dataset)
    val_acc = val_correct/len(val_loader.dataset)

    print(f"Epoch {epoch} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")

Epoch 1 | train loss 0.6864 acc 0.6044 | val loss 0.6513 acc 0.7582
Epoch 2 | train loss 0.6112 acc 0.8077 | val loss 0.5829 acc 0.8462
Epoch 3 | train loss 0.5437 acc 0.8654 | val loss 0.5178 acc 0.9231
Epoch 4 | train loss 0.4739 acc 0.9148 | val loss 0.4591 acc 0.9231
Epoch 5 | train loss 0.4213 acc 0.9203 | val loss 0.4072 acc 0.9121
Epoch 6 | train loss 0.3715 acc 0.9368 | val loss 0.3628 acc 0.9121
Epoch 7 | train loss 0.3266 acc 0.9368 | val loss 0.3260 acc 0.9231
Epoch 8 | train loss 0.2874 acc 0.9478 | val loss 0.2968 acc 0.9231
Epoch 9 | train loss 0.2650 acc 0.9451 | val loss 0.2713 acc 0.9341
Epoch 10 | train loss 0.2305 acc 0.9533 | val loss 0.2499 acc 0.9451
Epoch 11 | train loss 0.2053 acc 0.9560 | val loss 0.2325 acc 0.9451
Epoch 12 | train loss 0.1935 acc 0.9560 | val loss 0.2183 acc 0.9451
Epoch 13 | train loss 0.1803 acc 0.9615 | val loss 0.2067 acc 0.9451
Epoch 14 | train loss 0.1640 acc 0.9615 | val loss 0.1970 acc 0.9451
Epoch 15 | train loss 0.1583 acc 0.9698 | v

In [16]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor.to(device))
    test_loss = criterion(test_logits,y_test_tensor.to(device))
    test_acc = accuracy(test_logits,y_test_tensor.to(device))
print("Test kaybı:", test_loss.item())
print("Test doğruluğu:", test_acc.item())

Test kaybı: 0.1089426651597023
Test doğruluğu: 0.9736841917037964


#### LSTM ve GRU ile zaman serisi

In [17]:
t = np.arange(2000)
series = np.sin(0.05*t)+0.5*np.sin(0.011*t)+np.random.normal(0,0.1,len(t))
series = series.astype("float32")

In [18]:
WINDOW = 30

def make_windows(data,window):
    X,y = [],[]
    for i in range(len(data)-window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X)[...,np.newaxis],np.array(y)

X,y = make_windows(series,WINDOW)

In [19]:
split = int(0.8*len(X))
X_train,X_test = X[:split],X[split:]
y_train,y_test = y[:split],y[split:]
print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

X_train (1576, 30, 1)
X_test (394, 30, 1)
y_train (1576,)
y_test (394,)


In [20]:
def build_model(cell="lstm",units=32):
    rnn_layer = layers.LSTM(units) if cell=="lstm" else layers.GRU(units)
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],1)),
        rnn_layer,
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="mse",metrics=["mae"])
    return model

In [21]:
results = {}

for cell in ["lstm","gru"]:
    model = build_model(cell)
    model.fit(X_train,y_train,epochs=20,batch_size=32,validation_split=0.1,verbose=0)
    mse,mae = model.evaluate(X_test,y_test,verbose=0)
    results[cell] = {"MSE":mse,"MAE":mae}
    print(cell,"MSE:",mse,"MAE:",mae)

lstm MSE: 0.01197437196969986 MAE: 0.08540762215852737
gru MSE: 0.012566967867314816 MAE: 0.0875747874379158


#### IMDB duygu analizi

In [22]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBED_DIM = 128
LSTM_UNITS = 64

In [23]:
(X_train,y_train),(X_test,y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
print("Eğitim:", len(X_train), "yorum | Test:", len(X_test), "yorum")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Eğitim: 25000 yorum | Test: 25000 yorum


In [24]:
word_index = keras.datasets.imdb.get_word_index()
index_word = {index+3:word for word,index in word_index.items()}
index_word.update({0:"<PAD>",1:"<START>",2:"<OOV>"})
print("Örnek yorum:", " ".join(index_word.get(i,"?") for i in X_train[0][:40]), "....")
print("Etiket:", "olumlu" if y_train[0]==1 else "olumsuz")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Örnek yorum: <START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <OOV> is an amazing actor and now the same being director <OOV> father came ....
Etiket: olumlu


In [25]:
X_train = keras.utils.pad_sequences(X_train,maxlen=MAX_LEN,padding="post",truncating="post")
X_test = keras.utils.pad_sequences(X_test,maxlen=MAX_LEN,padding="post",truncating="post")

In [26]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE,output_dim=EMBED_DIM,mask_zero=True),
    layers.LSTM(LSTM_UNITS),
    layers.Dropout(0.2),
    layers.Dense(1,activation="sigmoid"),
])

In [27]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss",patience=2,restore_best_weights=True)
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=["accuracy"])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
history = model.fit(X_train,y_train,validation_split=0.2,epochs=5,batch_size=64,callbacks=[early_stopping],verbose=1)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 109s 340ms/step - accuracy: 0.7585 - loss: 0.4898 - val_accuracy: 0.8608 - val_loss: 0.3532
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 139s 331ms/step - accuracy: 0.8853 - loss: 0.2936 - val_accuracy: 0.8596 - val_loss: 0.3487
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 140s 323ms/step - accuracy: 0.9123 - loss: 0.2348 - val_accuracy: 0.7776 - val_loss: 0.7017
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 141s 322ms/step - accuracy: 0.8907 - loss: 0.2836 - val_accuracy: 0.8440 - val_loss: 0.3886


In [29]:
test_loss,test_acc = model.evaluate(X_test,y_test,verbose=0)
print("Test kaybı:", test_loss)
print("Test doğruluğu:", test_acc)

Test kaybı: 0.3727445602416992
Test doğruluğu: 0.8393599987030029


In [30]:
import re

def encode_text(text):
    words = re.sub(r"[^a-z0-9' ]"," ",text.lower()).split()
    ids = [1]
    for w in words:
        idx = word_index.get(w)
        ids.append(idx+3 if idx is not None and idx+3 < VOCAB_SIZE else 2)
    return keras.utils.pad_sequences([ids],maxlen=MAX_LEN,padding="post",truncating="post")

examples = [
    "This movie was absolutely wonderful, the acting was brilliant and I loved every minute.",
    "What a waste of time. The plot was boring and the characters were terrible.",
    "It was okay, not great but not bad either.",
]

for text in examples:
    prob = float(model.predict(encode_text(text),verbose=0)[0,0])
    label = "OLUMLU" if prob > 0.5 else "OLUMSUZ"
    print(label, round(prob,3), text)

OLUMLU 0.924 This movie was absolutely wonderful, the acting was brilliant and I loved every minute.
OLUMSUZ 0.005 What a waste of time. The plot was boring and the characters were terrible.
OLUMSUZ 0.22 It was okay, not great but not bad either.
